# Exercises XP Gold - RNNs in PyTorch on MNIST CSV

This is a guided notebook for the exercise on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear for key ideas.


## What you will learn
- Implement and train RNN, GRU, and LSTM with PyTorch
- Preprocess and load sequence datasets
- Evaluate models with accuracy
- Use Dataset and DataLoader for efficient IO


## What you will create
- A custom dataset class for MNIST CSV
- Three sequence models: SimpleRNN, SimpleGRU, SimpleLSTM
- A training loop and an evaluation function


In [3]:
import zipfile
import os
from google.colab import files
uploaded = files.upload()

zip_path = "/content/MNIST in CSV.zip"
extract_path = "/content/mnist_csv"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)


Saving MNIST in CSV.zip to MNIST in CSV.zip


['MNIST in CSV']

# Part 1: Setting up the environment

**As stated in the exercise**  
Install libraries, define hyperparameters, and set the device.

**PREFILLED**  
Imports, seed control, hyperparameters, device selection, and simple helpers.


In [4]:
# PREFILLED: just execute
import os, time, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters - you may adjust in the To-Do cell below
INPUT_SIZE   = 28   # number of features per time step for MNIST row-wise
SEQ_LEN      = 28   # number of time steps
HIDDEN_SIZE  = 128
NUM_LAYERS   = 1
NUM_CLASSES  = 10
LR           = 1e-3
EPOCHS       = 5
BATCH_SIZE   = 128

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

**To-Do (written):** Briefly justify the choice of treating a 28x28 image as a sequence of length 28 with input size 28. When would column-wise sequencing change anything in practice?


We treat a 28x28 image as a sequence of 28 time steps (rows), each containing 28 features (pixels).  
Column-wise sequencing would just change the order of input vectors but the recurrent model would still learn spatial dependencies across time steps.


**Learning point**  
A grayscale image can be read as a sequence by rows or columns. The model consumes vectors of size 28 across 28 steps. Information flow differs slightly by ordering, but both work because local spatial patterns still get exposed to the recurrent state.


# Part 2: Designing the neural network models

**As stated in the exercise**  
Implement SimpleRNN, SimpleGRU, and SimpleLSTM. Each ends with a fully connected layer mapping the last hidden state to 10 classes.

**PREFILLED**  
Model skeletons with clear To-Do regions.


In [16]:
# PREFILLED: just execute
class SimpleRNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.rnn using nn.RNN with batch_first=True
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len, input_size]
        out, h_n = self.rnn(x)
        # get last hidden state and map to logits
        out = self.fc(h_n[-1])
        return out


class SimpleGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.gru using nn.GRU with batch_first=True
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # run GRU and map last state to logits
        out, h_n = self.gru(x)
        out = self.fc(h_n[-1])
        return out


class SimpleLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.lstm using nn.LSTM with batch_first=True
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # run LSTM and map last hidden state to logits
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(h_n[-1])
        return out


**Learning point**  
RNN keeps one hidden. GRU adds gates that control update and reset. LSTM keeps a cell state with input, forget, and output gates. These gates help retain or discard information over time steps.


# Part 3: Creating a custom dataset class

**As stated in the exercise**  
Create `MnistCsvDataset` that loads the CSV, preprocesses, and yields items compatible with sequences.

Dataset format expectation: a CSV with `label` as the first column, followed by 784 pixel columns named `pixel0 ... pixel783` or unnamed numeric columns. If your CSV differs, adapt the loader.


In [8]:
# PREFILLED: just execute
class MnistCsvDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        self.y = df.iloc[:, 0].astype(np.int64).to_numpy()
        self.X = df.iloc[:, 1:].to_numpy(dtype=np.float32) / 255.0
        self.X = self.X.reshape(-1, 28, 28)  # [N, seq_len, input_size]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        seq = torch.tensor(self.X[idx], dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.long)
        return seq, label


**To-Do (code):** Instantiate training and test datasets from CSV files. Wrap them in DataLoaders.

Use variables below. Replace the paths with your local paths.


In [10]:
import os

# Vérifie le contenu exact du dossier
for root, dirs, files in os.walk("/content"):
    for f in files:
        if "mnist" in f.lower():
            print(os.path.join(root, f))


/content/MNIST in CSV.zip
/content/mnist_csv/MNIST in CSV/mnist_test.zip
/content/mnist_csv/MNIST in CSV/mnist_train.zip
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv


In [11]:
# To-Do: create datasets and loaders
import os
import zipfile
import torch
from torch.utils.data import DataLoader

# === 1️⃣ Extraction des inner zips ===
inner_dir = "/content/mnist_csv/MNIST in CSV"
extract_final = "/content/mnist_final"

os.makedirs(extract_final, exist_ok=True)

for fname in ["mnist_train.zip", "mnist_test.zip"]:
    zip_path = os.path.join(inner_dir, fname)
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_final)

print("✅ Extraction terminée. Fichiers extraits :", os.listdir(extract_final))

# === 2️⃣ Préparation des chemins ===
DATA_DIR = extract_final
TRAIN_CSV = os.path.join(DATA_DIR, "mnist_train.csv")
TEST_CSV = os.path.join(DATA_DIR, "mnist_test.csv")

# === 3️⃣ Instanciation des datasets ===
ds_train = MnistCsvDataset(TRAIN_CSV)
ds_test = MnistCsvDataset(TEST_CSV)

# === 4️⃣ DataLoaders ===
dataloader_args = dict(batch_size=128, num_workers=2, pin_memory=torch.cuda.is_available())
dl_train = DataLoader(ds_train, shuffle=True, **dataloader_args)
dl_test = DataLoader(ds_test, shuffle=False, **dataloader_args)

print("✅ Dataset prêt :")
print("Train samples :", len(ds_train))
print("Test samples  :", len(ds_test))


✅ Extraction terminée. Fichiers extraits : ['mnist_test.csv', 'mnist_train.csv']
✅ Dataset prêt :
Train samples : 60000
Test samples  : 10000


# Part 4: Training the model

**As stated in the exercise**  
Use CrossEntropyLoss and Adam. Implement a training loop that prints epoch loss.


In [12]:
# PREFILLED: training helpers
# PREFILLED: training helpers
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_batches = 0
    for xb, yb in loader:
        # To-Do: move xb, yb to device and ensure xb shape [B, SEQ_LEN, INPUT_SIZE]
        xb = xb.to(device)
        yb = yb.to(device)

        # To-Do: forward, compute loss, backward, step, zero_grad
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        # Accumulate total_loss
        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)


def fit(model, dl_train, dl_val, epochs, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_val = math.inf

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, dl_train, criterion, optimizer, device)
        val_loss = evaluate_loss(model, dl_val, criterion, device)
        dt = time.time() - t0
        print(f"epoch {epoch:02d} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - time_s {dt:.1f}")
        if val_loss < best_val:
            best_val = val_loss
    return model


**To-Do (code):** Implement `evaluate_loss` to compute average loss on a loader. Then construct one of the models, send to device, and call `fit`.


In [13]:
# To-Do: evaluation and training
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total += loss.item() * xb.size(0)
            n += xb.size(0)
    return total / max(1, n)


# === RNN MODEL TRAINING ===
rnn_model = SimpleRNNModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
rnn_model = fit(rnn_model, dl_train, dl_test, EPOCHS, device)


epoch 01 - train_loss 0.8973 - val_loss 0.4748 - time_s 4.5
epoch 02 - train_loss 0.3970 - val_loss 0.3107 - time_s 2.9
epoch 03 - train_loss 0.2666 - val_loss 0.2264 - time_s 2.9
epoch 04 - train_loss 0.2233 - val_loss 0.1929 - time_s 3.0
epoch 05 - train_loss 0.1914 - val_loss 0.1692 - time_s 3.8


# Part 5: Evaluating the model

**As stated in the exercise**  
Implement accuracy for train and test. Use the trained model to predict test labels and compute accuracy.


In [14]:
# To-Do: accuracy function
def accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(1, total)


print({
    'train_acc': accuracy(rnn_model, dl_train, device),
    'test_acc': accuracy(rnn_model, dl_test, device)
})


{'train_acc': 0.9550166666666666, 'test_acc': 0.9523}


**To-Do (code):** Repeat training and evaluation for `SimpleGRUModel` and `SimpleLSTMModel`. Compare the final test accuracies.


In [17]:
# === GRU MODEL TRAINING ===
gru_model = SimpleGRUModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
gru_model = fit(gru_model, dl_train, dl_test, EPOCHS, device)
print({
    'train_acc': accuracy(gru_model, dl_train, device),
    'test_acc': accuracy(gru_model, dl_test, device)
})

# === LSTM MODEL TRAINING ===
lstm_model = SimpleLSTMModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
lstm_model = fit(lstm_model, dl_train, dl_test, EPOCHS, device)
print({
    'train_acc': accuracy(lstm_model, dl_train, device),
    'test_acc': accuracy(lstm_model, dl_test, device)
})


epoch 01 - train_loss 0.7501 - val_loss 0.2926 - time_s 4.9
epoch 02 - train_loss 0.2118 - val_loss 0.1479 - time_s 3.0
epoch 03 - train_loss 0.1388 - val_loss 0.1184 - time_s 3.0
epoch 04 - train_loss 0.1007 - val_loss 0.1076 - time_s 3.0
epoch 05 - train_loss 0.0812 - val_loss 0.0792 - time_s 4.0
{'train_acc': 0.9792333333333333, 'test_acc': 0.9757}
epoch 01 - train_loss 0.6327 - val_loss 0.2276 - time_s 3.1
epoch 02 - train_loss 0.1816 - val_loss 0.1529 - time_s 3.0
epoch 03 - train_loss 0.1194 - val_loss 0.1003 - time_s 3.9
epoch 04 - train_loss 0.0941 - val_loss 0.0897 - time_s 3.0
epoch 05 - train_loss 0.0729 - val_loss 0.0911 - time_s 3.0
{'train_acc': 0.9787166666666667, 'test_acc': 0.9743}
